# Power-Flow Validation: Synthetic vs Real Grids (Status Quo)

Grid-model validation of the joint DSO A + DSO B run (`joint_setup.RUN_ID`): the status-quo power flow of the
synthetic pylovo grids against the real DSO grids over the same buildings. The pooled figure and table label both
providers as *Real* and *Synthetic*, so no DSO is named; the per-provider views are diagnostics only.

**Grid set.** A grid with non-converged power-flow hours in at least 1 % of the hours of any case is left out of
every case of its own group (`joint_setup.MIN_FAILED_SHARE`): its summaries would miss exactly its worst hours.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from gridexpand.analysis.expansion.notebook_workflow import PROVIDER_LABELS, load_powerflow_cutoff_comparison
from gridexpand.analysis.plotting.powerflow_asset_plots import (
    plot_cable_max_loading_ecdf,
    plot_powerflow_asset_cutoff_overview_static,
)
from gridexpand.analysis.powerflow.comparison_data import powerflow_distribution_similarity_summary
from joint_setup import (
    CASE_COLORS,
    CASE_DISPLAY,
    CURVE_Y_LIMITS,
    NETWORK_COLORS,
    OUTPUT_DIR,
    STAGE_LABELS,
    prepare_joint_analysis,
    stage_specs,
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)

context = prepare_joint_analysis()
PRE = STAGE_LABELS["pre"]
FIGURE_DIR = OUTPUT_DIR / "powerflow"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

## Readiness

The gate checks the raw run: `compact power-flow runs complete` fails as long as any grid has non-converged hours,
also the grids excluded below.

In [ ]:
display(context["powerflow_status"])
display(context["publication_gate"])
if not context["publication_ready"]:
    print("DIAGNOSTIC: at least one readiness check failed (see the gate above).")

## Non-converged grids

Every grid with non-converged hours in one of the cases. `excluded` marks the grids left out; the others stay in,
their few missing hours barely move the summaries.

In [ ]:
display(context["nonconverged"])
context["excluded_grids"]

In [ ]:
comparison = load_powerflow_cutoff_comparison(
    specs_by_source=stage_specs(context, PRE),
    stage_order=[PRE],
    excluded_grids=context["excluded_grids"],
)
profile = comparison["profile"]
display(comparison["coverage_summary"])
display(comparison["excluded_grids"])

## Distribution similarity

Synthetic minus real for the critical value of each grid asset (transformer and cable annual maximum loading,
annual minimum voltage); the pooled rows are the table *powerflow-distribution-summary*.

In [ ]:
def similarity(frame):
    return powerflow_distribution_similarity_summary(
        frame, group_col="network", synthetic_group="Synthetic", real_group="Real"
    )


tables = {"Pooled": similarity(profile)}
for provider in context["providers"]:
    label = PROVIDER_LABELS[provider]
    tables[label] = similarity(profile[profile["provider"] == label])
similarity_table = pd.concat(tables, names=["scope"]).reset_index(level=0)
similarity_table.to_csv(FIGURE_DIR / "powerflow_distribution_similarity.csv", index=False)
similarity_table

## Status-quo stress

Figure *powerflow-asset-percentiles*: mean over the retained grids by retained-grid cutoff (top) and the
distribution of each grid's critical asset at the P95 cutoff (bottom), pooled Real vs Synthetic.

In [ ]:
fig = plot_powerflow_asset_cutoff_overview_static(
    profile,
    group_col="comparison_stage",
    source_col="network",
    color_map=CASE_COLORS,
    group_labels=CASE_DISPLAY,
    asset_cutoff_percentile=0.99,
    asset_percentiles=(0.10, 0.25, 0.50, 0.75, 0.90, 1.0),
    filter_scope="grid",
    worst_asset_per_grid=True,
    curve_y_axis_limits=CURVE_Y_LIMITS,
    save_path=FIGURE_DIR / "asset-percentiles",
)
plt.show()

### Per provider (diagnostic; names the providers)

In [ ]:
for provider in context["providers"]:
    label = PROVIDER_LABELS[provider]
    fig = plot_powerflow_asset_cutoff_overview_static(
        profile[profile["provider"] == label],
        group_col="comparison_stage",
        source_col="network",
        color_map=CASE_COLORS,
        group_labels=CASE_DISPLAY,
        asset_cutoff_percentile=0.95,
        asset_percentiles=(0.10, 0.25, 0.50, 0.75, 0.90, 1.0),
        filter_scope="grid",
        curve_y_axis_limits=CURVE_Y_LIMITS,
        title=label,
        save_path=FIGURE_DIR / f"asset-percentiles-{provider}",
    )
    plt.show()

## Cable maximum loading ECDF

In [ ]:
plot_cable_max_loading_ecdf(
    profile,
    group_col="network",
    color_map=NETWORK_COLORS,
    title="Status-quo cable maximum loading",
    show=False,
)